# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a Metadata object
print("Loaded Dataset:\n")
print(f"Title: {metadata.name if hasattr(metadata, 'name') else metadata.__dict__.get('name')}</br>")
print(f"Description: {metadata.description if hasattr(metadata, 'description') else metadata.__dict__.get('description')}")

## 2. Data Overview
List available record sets (`cr:RecordSet`), with their `@id` and a sample of their fields. All entities are referenced by their `@id` for clarity and reproducibility.

In [ ]:
# Helper function: Safely display @id and field info for RecordSets
def get_record_sets(ds):
    if hasattr(ds.metadata, 'record_sets'):
        return ds.metadata.record_sets
    elif hasattr(ds.metadata, 'recordSet'):
        return ds.metadata.recordSet
    return []

record_sets = get_record_sets(dataset)
if not record_sets:
    print("No record sets found in the Croissant metadata.\nTry extracting record sets from the dataset as shown below.")
    # Fallback: Try the API directly instead of relying on incomplete @id lists
    print("Attempting to enumerate record sets via dataset API:")
    try:
        discovered_ids = set()
        # mlcroissant currently exposes record_set keys via dataset.records(record_set=...) only if you know the ID;
        # So use 'record_sets' method if available, otherwise document as an example with the main table
        print("Please refer to FAIR² documentation for full list of record sets. For this notebook, we'll use the main tabular record set.")
    except Exception as e:
        print(f"Dataset exploration error: {e}")
else:
    for rs in record_sets:
        print(f"Record set @id: {getattr(rs, '@id', getattr(rs, 'id', None))}")
        if hasattr(rs, 'fields'):
            field_ids = [getattr(f, '@id', None) for f in rs.fields[:5]]
            print(f"  Fields (first 5) @ids: {field_ids}")
        print()

# For demonstration, let's enumerate a few records from the primary record set.
# The main record set in FAIR² schema is usually '@id': 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034-table',
# but you can check the Croissant schema to confirm.
example_record_set_id = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034-table'
print(f"\nExample records for record set @id: {example_record_set_id}")
for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
    print(record)
    if i >= 2:
        break  # Show only a few for brevity

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. Always reference tabular sources by their `@id`, which for this dataset is likely:
```
https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034-table
```
You can add and extract more as needed.

In [ ]:
# List of record sets to extract, by their @id
record_set_ids = [
    'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034-table'
]

dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} rows and {len(df.columns)} columns from record set @id: {record_set_id}")

print("\nColumns in the loaded DataFrame:")
print(dataframes[record_set_ids[0]].columns.tolist())

# Show a preview
dataframes[record_set_ids[0]].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, and grouping.
All columns must be referenced by their `@id`.

Let's pick two fields for demonstration:
- For numeric filtering/normalization: `"MW"` (Molecular Weight, field @id usually like `https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034-table/MW`)
- For grouping: `"Accession"` (Unique protein ID, field @id usually like `https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034-table/Accession`)

Adjust @ids below to match the Croissant schema fields if needed.

In [ ]:
# Use the exact field @ids from the schema
table_id = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034-table'
df = dataframes[table_id]

# Field @ids (should be confirmed via Data Overview step if unsure)
mw_field = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034-table/MW'
accession_field = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034-table/Accession'

# Check actual columns present
print("Sample columns found:", df.columns[:10].tolist())

# Sometimes the @id is not the column header, but human names (e.g., 'MW'). Try fallback if necessary
if mw_field not in df.columns:
    for col in df.columns:
        # Try to find 'MW' (molecular weight) in columns
        if col.endswith('/MW') or col == 'MW':
            mw_field = col
            break
if accession_field not in df.columns:
    for col in df.columns:
        if col.endswith('/Accession') or col == 'Accession':
            accession_field = col
            break

if mw_field not in df.columns or accession_field not in df.columns:
    print("Relevant fields not found in DataFrame columns. Field @ids may differ from assumptions.")
else:
    # Drop missing values for analysis
    numeric_field = mw_field
    group_field = accession_field
    # Ensure numeric
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    # Filter proteins with MW > 50 kDa
    threshold = 50000
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Records with {numeric_field} > {threshold}:")
    print(filtered_df[[group_field, numeric_field]].head())

    # Normalize MW
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field}:")
    print(filtered_df[[group_field, numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by protein accession and compute mean MW
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped mean MW by {group_field} (first 5):")
    print(grouped_df.head())

## 5. Visualization
Visualize the Molecular Weight (MW) distribution for proteins with MW > 50,000.
You can explore more relationships or fields as appropriate.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if mw_field in filtered_df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(filtered_df[mw_field], kde=True, bins=30, color='skyblue')
    plt.xlabel('Molecular Weight (Da)')
    plt.ylabel('Count')
    plt.title('Distribution of MW (>50,000) across proteins')
    plt.tight_layout()
    plt.show()
else:
    print("Field for MW not present for visualization.")

## 6. Conclusion
In this notebook, we explored the FAIR² dataset's main protein abundance table using `mlcroissant`:
- Loaded metadata and extracted main tabular data with all fields referenced by their `@id` as defined in the Croissant schema.
- Performed exploratory data analysis: filtered proteins by molecular weight, normalized the MW distribution, and grouped by protein accession.
- Visualized the molecular weight distribution for large proteins.

**Next steps:** Continue the analysis by exploring other fields with their schema `@id`, join related record sets, or perform machine learning on the quantitative data.

_For more field and record set options, consult the full Croissant schema at the dataset source URL._